In [ ]:
import sys
sys.path.insert(0, "../..") # If issue with the duckify_simulation moduole check it position

USE_SIMULATOR = True  # <-- Toggle: True = Gazebo sim, False = real robot

if USE_SIMULATOR:
    from duckify_simulation import DuckifySim as ISCoin
    iscoin = ISCoin()
else:
    from URBasic import ISCoin
    ROBOT_IP = "10.30.5.158"  # UR3e1 (closest to window) / UR3e2: 10.30.5.159
    iscoin = ISCoin(host=ROBOT_IP, opened_gripper_size_mm=40)

from URBasic import Joint6D, TCP6D
from URBasic.waypoint6d import TCP6DDescriptor
from math import radians

rc = iscoin.robot_control  # shortcut

DuckifySim connected to container 'iscoin_simulator'


In [ ]:
# Joint angles
joints = rc.get_actual_joint_positions()
print(f"Joint positions: {joints}")

# TCP (tool center point) in Cartesian space
tcp = rc.get_actual_tcp_pose()
print(f"TCP pose:        {tcp}")

# TCP force (always 0 in simulator)
force = rc.force()
print(f"TCP force:       {force:.2f} N")

In [ ]:
# Reset any previous error
rc.reset_error()

# Save current position so we can return to it
start_joints = rc.get_actual_joint_positions()
print(f"Starting at: {start_joints}")

In [ ]:
iscoin.robot_control.reset_error()

# Move joints into positions
waypoints = [
    Joint6D.createFromRadians(1.1945080757141113, -1.126845733528473, 1.0483668486224573, -1.59875549892568, -1.5213754812823694, 1.046877384185791),
    Joint6D.createFromRadians(1.7649006843566895, -1.1278104347041626, 1.048335377370016, -1.5995241604247035, -1.4845483938800257, 1.0468727350234985),
    Joint6D.createFromRadians(1.5492383241653442, -1.4934492421201249, 1.5340636412249964, -1.7066379986205042, 0.101870097219944, 0.9368764162063599),
    Joint6D.createFromRadians(1.5487648248672485, -1.4930702310851593, 1.5343602339373987, -1.7022167644896449, -1.6295493284808558, 0.936877965927124),
    Joint6D.createFromRadians(0.4645763635635376, -1.4930564922145386, 1.5344775358783167, -1.7022696934142054, -1.629660431538717, 0.9369185566902161),
]
print(f'Joints are at {iscoin.robot_control.get_actual_joint_positions()} - going to {waypoints[0]}')
iscoin.robot_control.movej(waypoints[0], a = radians(80), v = radians(60))
print(f'Joints are at {iscoin.robot_control.get_actual_joint_positions()} - going to {waypoints[1]}')
iscoin.robot_control.movej(waypoints[1], a = radians(80), v = radians(60))
print(f'Joints are at {iscoin.robot_control.get_actual_joint_positions()} - going to {waypoints[2]}')
iscoin.robot_control.movej(waypoints[2], a = radians(80), v = radians(60))
print(f'Joints are at {iscoin.robot_control.get_actual_joint_positions()} - going to {waypoints[3]}')
iscoin.robot_control.movej(waypoints[3], a = radians(80), v = radians(60))
print(f'Joints are at {iscoin.robot_control.get_actual_joint_positions()} - going to {waypoints[4]}')
iscoin.robot_control.movej(waypoints[4], a = radians(80), v = radians(60))

In [ ]:
# Move to home position
home = Joint6D.createFromRadians(1.1945, -1.1268, 1.0484, -1.5988, -1.5214, 1.0469)
rc.movej(home, a=radians(80), v=radians(60))
print(f"At home: {rc.get_actual_joint_positions()}")

In [ ]:
# Move to a different joint position
target = Joint6D.createFromRadians(1.7649, -1.1278, 1.0483, -1.5995, -1.4845, 1.0469)
rc.movej(target, a=radians(80), v=radians(60))
print(f"At target: {rc.get_actual_joint_positions()}")

In [ ]:
# Move back home
rc.movej(home, a=radians(80), v=radians(60))
print(f"Back home: {rc.get_actual_joint_positions()}")

In [ ]:
# Make sure we're at home first
rc.movej(home, a=radians(80), v=radians(60))

tcp_start = rc.get_actual_tcp_pose()
print(f"TCP start: {tcp_start}")

In [ ]:
# Move 5cm in X
target1 = TCP6D.createFromMetersRadians(
    tcp_start.x + 0.05, tcp_start.y, tcp_start.z,
    tcp_start.rx, tcp_start.ry, tcp_start.rz,
)
rc.movel(target1, v=0.1)
print(f"After +5cm X: {rc.get_actual_tcp_pose()}")

In [ ]:
# Move 5cm in Y
target2 = TCP6D.createFromMetersRadians(
    tcp_start.x + 0.05, tcp_start.y + 0.05, tcp_start.z,
    tcp_start.rx, tcp_start.ry, tcp_start.rz,
)
rc.movel(target2, v=0.1)
print(f"After +5cm Y: {rc.get_actual_tcp_pose()}")

In [ ]:
# Move back to start
rc.movel(tcp_start, v=0.1)
print(f"Back to start: {rc.get_actual_tcp_pose()}")

In [ ]:
print(f"Back to start: {rc.get_actual_tcp_pose()}")

In [ ]:
# Go home and read TCP
rc.movej(home, a=radians(80), v=radians(60))
tcp0 = rc.get_actual_tcp_pose()
print(f"TCP before square: {tcp0}")

In [ ]:
L = 0.10  # 10cm side length
v = 0.05  # slow speed for visibility
blend = 0.01  # 1cm blend radius at corners

corners = [
    TCP6D.createFromMetersRadians(tcp0.x + L, tcp0.y,     tcp0.z, tcp0.rx, tcp0.ry, tcp0.rz),
    TCP6D.createFromMetersRadians(tcp0.x + L, tcp0.y + L, tcp0.z, tcp0.rx, tcp0.ry, tcp0.rz),
    TCP6D.createFromMetersRadians(tcp0.x,     tcp0.y + L, tcp0.z, tcp0.rx, tcp0.ry, tcp0.rz),
    TCP6D.createFromMetersRadians(tcp0.x,     tcp0.y,     tcp0.z, tcp0.rx, tcp0.ry, tcp0.rz),
]

waypoints = [
    TCP6DDescriptor(corners[0], v=v, r=blend),
    TCP6DDescriptor(corners[1], v=v, r=blend),
    TCP6DDescriptor(corners[2], v=v, r=blend),
    TCP6DDescriptor(corners[3], v=v),  # last point: no blend, stop exactly
]

print("Drawing square with movel_waypoints...")
rc.movel_waypoints(waypoints)
print(f"TCP after square: {rc.get_actual_tcp_pose()}")

In [ ]:
iscoin.close()
print("Done.")